In [ ]:
%%capture
%pip install pandas

In [ ]:
import pandas as pd
import re

In [ ]:
raw_data_path = "data/raw.csv"
raw = pd.read_csv(raw_data_path).drop(columns=['Reactions'])

In [ ]:
raw.head()

In [ ]:
len(raw)

### Remove users you don't want to be modeled (bots, low activitiy)

In [ ]:
# print out all users and their relative activity
raw["Author"].value_counts()

In [ ]:
# choose users to remove
remove_users = [""]

In [ ]:
# remove users
clean = raw[~raw["Author"].isin(remove_users)]
len(clean)

### Add a `has_media` flag for message with media

In [ ]:
URL_PATTERN = r"https?://\S+"

clean["has_media"] = (
    (clean["Attachments"].notna()) & (clean["Attachments"] != "") # has attachments in the Attachments column
) | ( # OR
    clean["Content"].str.contains(URL_PATTERN, case=False, na=False) # has URLs in the Content column
)
clean = clean.drop(columns=["Attachments"])

In [ ]:
clean.head()

### Add a `has_text` flag for message with content (for message with media AND content)

In [ ]:
clean["has_text"] = (
    clean["Content"]
    .fillna("")
    .str.replace(URL_PATTERN, "", regex=True) # remove URLs
    .str.strip()
    .str.len() > 0
)

In [ ]:
clean.head()

### Change mentions from @nickname to @username

In [ ]:
# nickname/server name to discord username mapping
nickname_to_username = {
    "nickname (any alias you want recognized)": "discord given username",
    "Timothee": "RealChalamet"
}

In [ ]:
pattern = re.compile("|".join(map(re.escape, nickname_to_username.keys())))
pattern

In [ ]:
clean["Content"] = clean["Content"].fillna("").str.replace(
    pattern,
    lambda m: nickname_to_username[m.group(0)],
    regex=True
)